# Tesseract OCR on Whole Image (Step-by-Step)

Target image: `Icdar2013\\Challenge2_Test_Task12_Images\\img_1.jpg`

## Step 1) Initialize Tesseract

In [1]:
from pathlib import Path
import cv2
import pytesseract

# Set this if Tesseract is not in PATH
pytesseract.pytesseract.tesseract_cmd = r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"

OCR_LANG = "eng"
OCR_CONFIG = "--oem 3 --psm 6"

## Step 2) Load the full image

In [4]:
image_path = Path(r"Icdar2013\\Challenge2_Test_Task12_Images\\img_1.jpg")
if not image_path.exists():
    image_path = Path(r"..\\..\\Icdar2013\\Challenge2_Test_Task12_Images\\img_1.jpg")

if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path.resolve()}")

img_bgr = cv2.imread(str(image_path))
if img_bgr is None:
    raise RuntimeError("cv2.imread failed. Check image path/permissions.")

print(f"Step 2 complete: Loaded {image_path.resolve()}")
print(f"Image shape: {img_bgr.shape}")

Step 2 complete: Loaded D:\Handheld OCR Translator\Icdar2013\Challenge2_Test_Task12_Images\img_1.jpg
Image shape: (1280, 960, 3)


## Step 3) Preprocess whole image

In [5]:
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3, 3), 0)
_, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print("Step 3 complete: grayscale + blur + Otsu threshold applied.")

Step 3 complete: grayscale + blur + Otsu threshold applied.


## Step 4) Run Tesseract on whole image

In [6]:
raw_text = pytesseract.image_to_string(binary, lang=OCR_LANG, config=OCR_CONFIG)
data = pytesseract.image_to_data(
    binary,
    lang=OCR_LANG,
    config=OCR_CONFIG,
    output_type=pytesseract.Output.DATAFRAME
)

print("Step 4 complete: OCR finished.")
print("Raw OCR preview:\n")
print(raw_text[:1000])

Step 4 complete: OCR finished.
Raw OCR preview:

Ti-edness
Kiils

e we fy ly ved

coulésave™
“our life



## Step 5) Post-process OCR text

In [7]:
df = data.copy()
df = df.dropna(subset=["text", "conf"])
df = df[df["text"].astype(str).str.strip() != ""]
df = df[df["conf"] >= 60]

clean_text = " ".join(df["text"].astype(str).tolist()).strip()
if not clean_text:
    clean_text = raw_text.strip()

print("Step 5 complete: text cleaned.")
print(clean_text[:1000])

Step 5 complete: text cleaned.
Ti-edness “our life


## Step 6) Translate / display / save

In [8]:
translated_text = ""
try:
    from deep_translator import GoogleTranslator
    translated_text = GoogleTranslator(source="auto", target="en").translate(clean_text) if clean_text else ""
except Exception:
    translated_text = "(Translation skipped: install deep-translator or check internet.)"

print("\n--- Final OCR Text ---\n")
print(clean_text[:2000])

print("\n--- Translated Text ---\n")
print((translated_text or "")[:2000])

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

(out_dir / "img_1_ocr.txt").write_text(clean_text, encoding="utf-8")
(out_dir / "img_1_translated.txt").write_text(translated_text or "", encoding="utf-8")

print("\nStep 6 complete: files saved.")
print(f"- {(out_dir / 'img_1_ocr.txt').resolve()}")
print(f"- {(out_dir / 'img_1_translated.txt').resolve()}")


--- Final OCR Text ---

Ti-edness “our life

--- Translated Text ---

(Translation skipped: install deep-translator or check internet.)

Step 6 complete: files saved.
- D:\Handheld OCR Translator\handheld-ocr-translator\tesseract_tutorial\outputs\img_1_ocr.txt
- D:\Handheld OCR Translator\handheld-ocr-translator\tesseract_tutorial\outputs\img_1_translated.txt
